# Tennis Match Dashboard Tutorial

This tutorial walks through building a small dashboard for 2023 ATP tennis match data, the same way we built the job salaries dashboard: pandas for the data, a class to hold the analysis, a second class that adds charts on top of it, and one small algorithm.

## What You Will Build

By the end of this notebook you will have a `TennisDashboard` class that summarizes the season and a `VisualTennisDashboard` class that extends it with charts.

- Import pandas and matplotlib
- Read the match data from a CSV file
- Build a base class that analyzes the data
- Build a child class that inherits from the base class and adds charts
- Create an object from the child class and run its methods

## The Dataset

The file `data/atp_matches_2023.csv` contains every men's ATP tour singles match from the 2023 season: 2,986 matches in total, including the tournament name, surface, round, match length in minutes, and the names and rankings of both players. It was compiled by Jeff Sackmann's [tennis_atp](https://github.com/JeffSackmann/tennis_atp) project and is shared under a Creative Commons Attribution NonCommercial ShareAlike 4.0 license.

Place this file in a `data` folder next to this notebook, the same way the job salaries dashboard used `data/ds_salaries.csv`.

## Step 1: Import the Libraries

Just like the salaries dashboard, we need pandas to work with the data and matplotlib to draw the charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Step 2: Read the CSV

Load the file with `pd.read_csv`, then keep only the columns the dashboard actually needs.

In [ ]:
df = pd.read_csv("data/atp_matches_2023.csv")

df = df[[
    "tourney_name", "surface", "round", "minutes",
    "winner_name", "loser_name", "winner_rank", "loser_rank"
]]

A handful of matches are missing a `minutes` value, usually because a player retired early or the timing was not recorded, so we drop those rows before doing any math with match length.

In [ ]:
df = df.dropna(subset=["minutes", "surface"])
df.head()

## Step 3: Build the Base Class

This works the same way as `SalaryDashboard` did. The class stores the DataFrame on `self.df` and each method is one view of the data:

- `summary` – headline numbers for the season
- `top_players` – a leaderboard of match wins
- `average_duration_by_surface` and `average_duration_by_round` – two grouped averages
- `surface_breakdown` – how many matches were played on each surface
- `search_players` – scans the data one row at a time, the same **linear search** pattern used to search job titles

In [ ]:
class TennisDashboard:
    def __init__(self, dataframe):
        self.df = dataframe

    def summary(self):
        print(f"Matches: {len(self.df)}")
        print(f"Surfaces: {', '.join(self.df['surface'].unique())}")
        print(f"Average match length: {self.df['minutes'].mean():.0f} minutes")
        print(f"Longest match: {self.df['minutes'].max():.0f} minutes")
        print(f"Players with at least one win: {self.df['winner_name'].nunique()}")

    def top_players(self, n=10):
        return self.df["winner_name"].value_counts().head(n)

    def average_duration_by_surface(self):
        order = ["Hard", "Clay", "Grass"]
        return self.df.groupby("surface")["minutes"].mean().reindex(order)

    def average_duration_by_round(self):
        order = ["R128", "R64", "R32", "R16", "QF", "SF", "F"]
        return self.df.groupby("round")["minutes"].mean().reindex(order)

    def surface_breakdown(self):
        return self.df["surface"].value_counts()

    def search_players(self, keyword):
        keyword = keyword.lower()
        matches = []
        for _, row in self.df.iterrows():
            if keyword in row["winner_name"].lower():
                matches.append(row)
        return pd.DataFrame(matches).sort_values("minutes", ascending=False)

## Step 4: Build the Child Class

`VisualTennisDashboard` inherits from `TennisDashboard`, so it already has `summary`, `top_players`, and every other method for free. It only needs to add the chart methods, the same way `VisualSalaryDashboard` added charts on top of `SalaryDashboard`.

In [ ]:
class VisualTennisDashboard(TennisDashboard):
    def plot_top_players(self, n=10):
        data = self.top_players(n)
        plt.figure(figsize=(8, 5))
        data.sort_values().plot(kind="barh", color="#4C72B0")
        plt.xlabel("Matches won")
        plt.title(f"Top {n} Players by Match Wins, 2023")
        plt.tight_layout()
        plt.show()

    def plot_by_surface(self):
        data = self.average_duration_by_surface()
        plt.figure(figsize=(6, 4))
        data.plot(kind="bar", color="#55A868")
        plt.ylabel("Average match length (minutes)")
        plt.title("Average Match Length by Surface")
        plt.tight_layout()
        plt.show()

    def plot_by_round(self):
        data = self.average_duration_by_round()
        plt.figure(figsize=(6, 4))
        data.plot(kind="line", marker="o", color="#C44E52")
        plt.ylabel("Average match length (minutes)")
        plt.title("Average Match Length by Round")
        plt.tight_layout()
        plt.show()

    def plot_surface_breakdown(self):
        data = self.surface_breakdown()
        plt.figure(figsize=(5, 5))
        data.plot(kind="pie", autopct="%1.0f%%")
        plt.ylabel("")
        plt.title("Matches Played by Surface")
        plt.tight_layout()
        plt.show()

## Step 5: Instantiate the Dashboard and Run It

Create one `VisualTennisDashboard` object and reuse it for every method call. This is the same pattern as `dashboard = VisualSalaryDashboard(df)` in the salaries project.

Running `summary()` should print 2,754 matches across three surfaces, an average match length of about 119 minutes, and 226 different players with at least one win.

In [ ]:
dashboard = VisualTennisDashboard(df)
dashboard.summary()

### Top players by match wins

In [ ]:
dashboard.plot_top_players()

### Average match length by surface

In [ ]:
dashboard.plot_by_surface()

### Average match length by round

In [ ]:
dashboard.plot_by_round()

### Matches played by surface

In [ ]:
dashboard.plot_surface_breakdown()

### Searching for a player

Try `search_players` with a different last name and see who comes back.

In [ ]:
dashboard.search_players("djokovic").head(10)

## Wrap Up

This tutorial covers the same three ideas as the salaries dashboard, applied to a different dataset: pandas for cleaning and summarizing the data, a base class plus a child class for organizing the code, and one algorithm applied to a real question instead of a toy example.

### Try it yourself

- Try adding a method for average ranking of players who win in straight sets
- Try a method that compares average match length across different tournament levels
- Try rebuilding `search_players` so it can search by either winner or loser

In [ ]:
# Your extensions here